# CDA training with league-based self-play

Runs unchanged on a **Colab** VM and in the **docker/ml/dockerfile_ray_torch**
image. Set `PLATFORM` and `USE_GPU` in the next cell (both default to `auto`);
everything they imply is read from `config/runtime_profiles.json`.

| | Colab | Docker |
|---|---|---|
| Setup | mounts Drive, installs the pinned packages, asks for one restart | nothing - the image already has them |
| Repo | wherever you put it in Drive (`COLAB_REPO_PATH`) | `/workspace/code` |
| GPU | Runtime > Change runtime type > GPU | `docker run --gpus all ...` |
| Checkpoints | the Drive-backed repo, so a disconnect is recoverable | the repo, i.e. your bind-mounted working tree |
| Episode record (Parquet) | the VM's local disk - ~34 MB per 4,096-step episode is too much for Drive | the repo |

Without a GPU either way, the `cpu` profile is selected automatically and the
run continues on one core.

The configuration and loop live in
`gym_continuousDoubleAuction/train/train.py`, and every value they use lives in
`config/`, so the same run is reproducible headless:

```
python -m gym_continuousDoubleAuction.train.train
```

This notebook is a thin driver. Training *behaviour* is edited in `train.py`;
training *values* in `config/train_config.json`; *where a run executes* in
`config/runtime_profiles.json`. None of it belongs here.

In [9]:
# ============================ runtime parameters ============================
# The only two knobs in this notebook. Everything they imply - CPU/GPU counts,
# package pins, output paths - is read from config/runtime_profiles.json.

PLATFORM = 'auto'   # 'auto' | 'colab' | 'docker' | 'local'
USE_GPU = 'auto'    # 'auto' | True | False

# Colab bootstrap only. This is the one path that cannot come from config,
# because it is how config is found: the repo has to be located before
# runtime_profiles.json inside it can be read. Ignored off Colab.
COLAB_DRIVE_MOUNT = '/content/gdrive'
COLAB_REPO_PATH = '/content/gdrive/MyDrive/Colab Notebooks/MARL/gym-continuousDoubleAuction'
# ============================================================================

import json
import os
import re
import subprocess
import sys

# This cell asks one question only - "is this Colab?" - because the answer
# decides whether the package needs to be *made* importable. `runtime.py` does
# the real platform resolution in the next cell, once it can be imported.
IS_COLAB = PLATFORM == 'colab' or (
    PLATFORM == 'auto'
    and ('COLAB_RELEASE_TAG' in os.environ or 'google.colab' in sys.modules)
)


def _missing(specs):
    """Which of `specs` are absent, or present at the wrong pinned version."""
    from importlib.metadata import PackageNotFoundError, version

    out = []
    for spec in specs:
        name = re.split(r'[<>=!\[]', spec, 1)[0].strip()
        pinned = spec.split('==')[1] if '==' in spec else None
        try:
            installed = version(name)
        except PackageNotFoundError:
            out.append(spec)
            continue
        if pinned and installed != pinned:
            out.append(spec)
    return out


if IS_COLAB:
    from google.colab import drive

    drive.mount(COLAB_DRIVE_MOUNT)
    os.chdir(COLAB_REPO_PATH)
    sys.path.insert(0, COLAB_REPO_PATH)

    # Readable now that the repo is the working directory. Plain json, not
    # config_loader: importing the package needs gymnasium, which is what we
    # are about to install.
    with open('config/runtime_profiles.json') as fh:
        _pkgs = json.load(fh)['platforms']['colab']['pip_packages']

    _needed = _missing(_pkgs)
    if _needed:
        print('installing:', ' '.join(_needed))
        subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *_needed],
                       check=True)
        print('\n' + '=' * 72)
        print('Installed. Runtime > Restart session, then run this cell again.')
        print('A restart is required because the install moves packages Colab')
        print('has already imported; this cell is a no-op the second time.')
        print('=' * 72)
        raise SystemExit('restart required')
    print('colab bootstrap : packages already present')
else:
    print('colab bootstrap : skipped (not Colab)')

print('working dir     :', os.getcwd())

colab bootstrap : skipped (not Colab)
working dir     : /workspace/code


In [10]:
from gym_continuousDoubleAuction.train import runtime

# Exports RAY_DEBUG_DISABLE_MEMORY_MONITOR and anything else in
# tunable_constants.json's runtime_env_vars group. Must happen before ray is
# imported, which is why it is above the import and not beside it.
print('env      :', runtime.apply_env_vars())

import ray
import torch

from gym_continuousDoubleAuction.train.train import (
    TrainConfig,
    configure_run_logging,
    train,
)

rt = runtime.resolve(platform=PLATFORM, use_gpu=USE_GPU)
runtime.chdir_to_repo(rt)

print('ray      :', ray.__version__)
print('torch    :', torch.__version__, '| CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU      :', torch.cuda.get_device_name(0))
print()
print(runtime.summary(rt))

env      : {'RAY_DEBUG_DISABLE_MEMORY_MONITOR': 'True'}
ray      : 2.56.1
torch    : 2.13.0+cu130 | CUDA available: True
GPU      : NVIDIA GeForce RTX 4060

platform         : docker
hardware profile : gpu
ray.init         : {'ignore_reinit_error': True, 'include_dashboard': False, 'num_cpus': 2, 'num_gpus': 1}
working dir      : /workspace/code


### Configuration

Two files, two jobs:

* **`config/train_config.json`** - what the run *does*: agents, batch sizes,
  reward coefficients, iteration count. Identical on every machine.
  `TrainConfig` in `train.py` is the schema; the file holds every value.
* **`config/runtime_profiles.json`** - where the run *executes*: the `gpu` set
  (2 CPUs + 1 GPU) and the `cpu` set (1 CPU, no GPU), plus per-platform output
  paths. Selected by `PLATFORM` / `USE_GPU` above and resolved in `runtime.py`.

Change a knob by editing the file it belongs to, not this cell. An argument
passed to `TrainConfig(...)` shadows `train_config.json` for that one key - and
also shadows an alternative config tree pointed at by `$CDA_CONFIG_DIR` - which
is how a notebook run silently drifts from the headless one.

`train_batch_size` and `checkpoint_dir` are absent from both files on purpose:
they are derived (`max_step * num_episodes_per_iter`, and `log_base_dir/chkpt`)
and are computed below rather than stored twice.

In [11]:
from gym_continuousDoubleAuction.config_loader import config_dir

# No arguments: every training value is read from config/train_config.json.
base = TrainConfig()
# The resolved runtime then overlays resource counts and output paths, which
# are the only things that differ between Colab and the docker image.
cfg = runtime.apply(base, rt)
# Attaches this run's log files under cfg.run_dir and exports the level and
# directory for the workers ray.init is about to start. Before ray.init on
# purpose: a worker started first would inherit neither.
configure_run_logging(cfg)

print('config dir       :', config_dir())
print()
print('agents           :', cfg.num_agents, f'({cfg.num_trained_agents} trained)')
print('max step         :', cfg.max_step, '| episodes per iter:', cfg.num_episodes_per_iter)
print('train through it.:', cfg.num_iters, '| checkpoint every:', cfg.chkpt_freq, '| keep:', cfg.chkpt_keep)
# num_iters is the iteration to train THROUGH, so a restored run finishes it
# rather than repeating it. See doc/18_configuration.md 5.1.
print('restore          :', cfg.is_restore, '|',
      cfg.restore_path or 'newest checkpoint under the dir below')
print()
print(f'--- {rt.hardware} profile + {rt.platform} paths ---')
for name, (old, new) in runtime.changed_fields(base, cfg).items():
    print(f'  {name:<22} {old!r} -> {new!r}')
print()
# Derived, not stored: max_step * num_episodes_per_iter, and log_base_dir/chkpt.
print('train batch size :', cfg.train_batch_size)
# The whole batch has to be sampled within this, or the iteration discards its
# rollouts and trains on nothing. RLlib's own default of 60s is far too low for
# a 4096-step episode of Python order-book matching.
print('sample timeout   :', cfg.sample_timeout_s, 's')
print('run dir          :', cfg.run_dir)
# Outside run_dir deliberately, so a restore still finds the checkpoints an
# earlier run wrote. See doc/11 1.11.
print('checkpoint dir   :', cfg.checkpoint_dir)
print('episode data dir :', cfg.episode_data_dir)
# num_gpus_per_learner as actually applied: forced to 0 when CUDA is absent.
# With num_learners=0 the learner is in-process, so any value > 0 puts it on
# the GPU - the fraction is only a Ray resource request for *remote* learners.
print('gpus per learner :', cfg.resolved_gpus_per_learner())

2026-09-19 08:00:55 INFO    pid=44 iter=10 gym_continuousDoubleAuction.train.train: run id: run_20260919_080055_e783 (/workspace/code/results/run_20260919_080055_e783)
2026-09-19 08:00:55 INFO    pid=44 iter=10 gym_continuousDoubleAuction.train.train: run log: /workspace/code/results/run_20260919_080055_e783/run.log
2026-09-19 08:00:55 INFO    pid=44 iter=10 gym_continuousDoubleAuction.train.train: episode record: /workspace/code/episode_data/run_20260919_080055_e783
config dir       : /workspace/code/config

agents           : 8 (2 trained)
max step         : 4096 | episodes per iter: 4
train through it.: 16 | checkpoint every: 2 | keep: 3
restore          : False | newest checkpoint under the dir below

--- gpu profile + docker paths ---
  num_env_runners        0 -> 2
  num_cpus_per_env_runner 0.25 -> 1.0
  num_gpus_per_learner   0.25 -> 1.0

train batch size : 16384
sample timeout   : 600.0 s
run dir          : /workspace/code/results/run_20260919_080055_e783
checkpoint dir   : /wo

### Train

In [12]:
ray.shutdown()
# num_cpus / num_gpus come from the hardware profile, so Ray is told about
# exactly the machine the profile was written for rather than whatever the
# host happens to have.
# ray_init_kwargs() rather than the stored dict: it merges in $CDA_LOG_LEVEL
# and $CDA_LOG_DIR as a runtime_env, which is the only way they reach workers
# on a cluster this process did not start.
# `cfg` is passed so this also applies ray.LoggingConfig, which is what makes
# configure_run_logging's decision to turn propagation off the right one -
# without it, propagation is off and nothing takes its place.
ray.init(**runtime.ray_init_kwargs(rt, cfg))

# train() runs every iteration through cfg.num_iters and returns the last
# result alongside the algorithm, so inspecting the league below costs nothing.
algo, result = train(cfg)

2026-09-19 08:00:58,495	WARNING services.py:2248 -- WARNING: The object store is using /tmp/ray instead of /dev/shm because /dev/shm has only 2147483648 bytes available. This will harm performance! You may be able to free up space by deleting files in /dev/shm. If you are inside a Docker container, you can increase /dev/shm size by passing '--shm-size=4.44gb' to 'docker run' (or add it to the run_options list in a Ray cluster config). Make sure to set this to more than 30% of available RAM. timestamp_ns=1789804858495682911


2026-09-19 08:01:01 INFO    pid=44 iter=10 gym_continuousDoubleAuction.train.policy.policy_handler: modules: ['policy_0', 'policy_1', 'policy_2', 'policy_3', 'policy_4', 'policy_5', 'policy_6', 'policy_7'] | trainable: ['policy_0', 'policy_1'] (encoder mlp) | frozen random baselines: ['policy_2', 'policy_3', 'policy_4', 'policy_5', 'policy_6', 'policy_7']


2026-09-19 08:01:01,676	WARNING algorithm_config.py:2507 -- DeprecationWarning: `config.training(learner_class=..)` has been deprecated. Use `config.learners(learner_class=..)` instead. This will raise an error in the future! job_id=01000000 worker_id=01000000ffffffffffffffffffffffffffffffffffffffffffffffff node_id=334837d7c31b59111dbd551c6d91db1bfd95b5f515e7d07661dc9095 timestamp_ns=1789804861676629712
2026-09-19 08:01:01,677	WARNING algorithm_config.py:2530 -- DeprecationWarning: `config.training(learner_config_dict=..)` has been deprecated. Use `config.learners(learner_config_dict=..)` instead. This will raise an error in the future! job_id=01000000 worker_id=01000000ffffffffffffffffffffffffffffffffffffffffffffffff node_id=334837d7c31b59111dbd551c6d91db1bfd95b5f515e7d07661dc9095 timestamp_ns=1789804861677279386


2026-09-19 08:01:01 INFO    pid=44 iter=10 gym_continuousDoubleAuction.train.train: starting from scratch
2026-09-19 08:01:01 WARNING pid=44 iter=10 gym_continuousDoubleAuction.train.train: /workspace/code/results/chkpt already holds 3 checkpoint(s) from an earlier run, newest first: iter_00008, iter_00006, iter_00004. This run is not restoring, so they are left alone - but until it passes their iteration numbers, a --restore would pick one of them over anything written here. Move or delete them, or point log_base_dir somewhere new.
2026-09-19 08:01:06 INFO    pid=44 iter=10 gym_continuousDoubleAuction.train.train: policy_0: encoder mlp | 259,616 parameters (259,616 trainable)
2026-09-19 08:01:06 INFO    pid=44 iter=10 gym_continuousDoubleAuction.train.train: policy_1: encoder mlp | 259,616 parameters (259,616 trainable)


2026-09-19 08:01:06,622	WARNING rl_module.py:463 -- DeprecationWarning: `RLModule(config=[RLModuleConfig object])` has been deprecated. Use `RLModule(observation_space=.., action_space=.., inference_only=.., model_config=.., catalog_class=..)` instead. This will raise an error in the future! job_id=01000000 worker_id=7c26939ce22eb7c86188a2130da4a3209c01f20886731db8be9e9236 node_id=334837d7c31b59111dbd551c6d91db1bfd95b5f515e7d07661dc9095 actor_id=11ee759ea5a9b466bfc4b04001000000 task_id=ffffffffffffffff11ee759ea5a9b466bfc4b04001000000 task_name=MultiAgentEnvRunner.__init__ task_func_name=ray.rllib.env.multi_agent_env_runner.MultiAgentEnvRunner.__init__ timestamp_ns=1789804866622152311


2026-09-19 08:01:18 INFO    pid=76094 iter=1 gym_continuousDoubleAuction.envs.exchg.liquidation_helper: liquidation (close) of agent_3 at t_step 2843: position -4368, NAV 453032.0, mark 351.5; 651 closed in the book (band 455), 3717 by ADL, 0 steps left
2026-09-19 08:01:18 INFO    pid=76094 iter=1 gym_continuousDoubleAuction.envs.exchg.liquidation_helper: liquidation (close) of agent_6 at t_step 2853: position -2917, NAV 309760.0, mark 363.0; 303 closed in the book (band 469), 2614 by ADL, 0 steps left
2026-09-19 08:01:22 INFO    pid=76094 iter=1 gym_continuousDoubleAuction.train.callbk.league_based_self_play_callback: Episode 1e6cd588606d4678b0c3ec023328bad7 NAV verification
  agent_0 NAV: 1,033,387.50
  agent_1 NAV: 1,823,221.50
  agent_2 NAV: 1,088,158.50
  agent_3 NAV: 359,211.50
  agent_4 NAV: 1,004,265.00
  agent_5 NAV: 1,547,592.00
  agent_6 NAV: 299,079.50
  agent_7 NAV: 845,084.50
  Total NAV: 8,000,000.00
  Expected total initial cash: 8,000,000.00
  Conserved (within 1e-06)


### Inspect the league

`module_episode_returns_mean` is keyed by real ModuleID, so `champion_*` entries
are the frozen snapshots and `policy_0`/`policy_1` are the learners.

This reads the `result` cell 6 returned - the final iteration's own metrics.
Calling `algo.train()` here instead would run *another* full iteration for its
return value, outside the checkpointing `train()` does.

An empty table means the run trained on nothing: if `env_runners` is missing
from the result, the env runners did not deliver `train_batch_size` steps
within `sample_timeout_s` and their partial rollouts were discarded. The
iteration log in cell 6 says so directly.

In [13]:
from ray.rllib.utils.metrics import ENV_RUNNER_RESULTS

# No algo.train() here: `result` is the last iteration's, returned by train().
returns = result.get(ENV_RUNNER_RESULTS, {}).get('module_episode_returns_mean', {})

if returns:
    for module_id, value in sorted(returns.items()):
        print(f'{module_id:<14} {value:>14,.2f}')
else:
    print('no module returns in the final result - the run sampled nothing.')
    print('result keys:', sorted(result))

champion_1                nan
champion_2              -1.72
champion_3                nan
champion_4              -1.70
champion_5              -2.41
champion_6              -0.56
champion_7               1.25
policy_0                 1.77
policy_1                 1.47
policy_2                -1.17
policy_3                -1.75
policy_4                  nan
policy_5                  nan
policy_6                -1.74
policy_7                  nan


In [14]:
# The whole result dict of the final iteration, for anything not in the table
# above: timers, learner stats, fault_tolerance.
result

{'timers': {'training_iteration': 60.1504904854664,
  'restore_env_runners': 7.852266238012426e-06,
  'training_step': 60.15030085007741,
  'env_runner_sampling_timer': 35.418682157434944,
  'learner_update_timer': 24.641836854678854,
  'synch_weights': 0.007099136192562255,
  'synch_env_connectors': 0.002048330108373252},
 'env_runners': {'env_reset_timer': np.float64(0.0005179595000299742),
  'module_episode_returns_mean': {'policy_4': nan,
   'policy_5': nan,
   'policy_7': nan,
   'policy_2': np.float64(-1.1650184000000143),
   'policy_0': np.float64(1.765738349999974),
   'policy_1': np.float64(1.4677885249999707),
   'policy_6': np.float64(-1.742039800000023),
   'policy_3': np.float64(-1.752893050000022),
   'champion_1': nan,
   'champion_2': np.float64(-1.7225600000000312),
   'champion_3': nan,
   'champion_4': np.float64(-1.6965521999999749),
   'champion_5': np.float64(-2.413468349999954),
   'champion_6': np.float64(-0.5640485999999962),
   'champion_7': np.float64(1.25048

In [15]:
ray.shutdown()